Movies Table

In [1]:
import pandas as pd
import re

0. File Paths

In [2]:
META_PATH = "metaClean43Brightspace.xlsx"
SALES_PATH = "sales.xlsx"
OUT = "."

In [3]:
def fix_mojibake(raw):
    """ Mojibake is garbled unreadable text that happens when
    a computer miscodes a file. This cleans that up
    """
    if pd.isna(raw):
        return
    s = str(raw)
    try:
        return s.encode("latin1").decode("utf-8")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return s 

In [4]:
def to_int_str(series):
    return series.map(lambda v: "" if pd.isna(v) else str(int(v)))

1. Movies 

In [5]:
print("Loading metaClean and sales")
meta = pd.read_excel(META_PATH)
sales = pd.read_excel(SALES_PATH)

Loading metaClean and sales


In [6]:
meta = meta.reset_index(drop=True)
meta["title"] = meta["title"].apply(fix_mojibake)
meta["title_norm"] = meta["title"].astype(str).str.strip().str.lower()
"""This makes sure that everything is equal"""

meta["year"] = pd.to_datetime(meta["RelDate"], errors="coerce").dt.year

In [7]:
sales = sales.copy()
sales["title_norm"] = sales["title"].astype(str).str.lower().str.strip()
"""This de-duplicates the sales side on (title_norm, year), 
before joining since some keys match more than one row
"""
sales_dedup = sales.drop_duplicates(subset=["title_norm", "year"], keep="first")

In [8]:
movies = meta.merge(
    sales_dedup[["title_norm", "year", "production_budget", "worldwide_box_office"]],
    on=["title_norm", "year"], how="left", indicator=True
)
movies["movie_id"] = range(1, len(movies) + 1)

In [9]:
meta["_rel_date"] = pd.to_datetime(meta["RelDate"], errors="coerce")
meta["_month_day"] = meta["_rel_date"].apply(lambda d: (d.month, d.day) if pd.notna(d) else None)
movies["_month_day"] = meta["_month_day"]


1b. Separate out the same title, different year

In [10]:
sales_years_by_title = (
    sales_dedup.groupby("title_norm")["year"]
    .apply(lambda s: sorted(set(s.dropna().astype(int).tolist())))
)
matched_years_by_title = (
    movies[movies["_merge"] == "both"]
    .groupby("title_norm")["year"]
    .apply(lambda s: set(s.dropna().astype(int).tolist()))
    .to_dict()
)

In [11]:
def matching_years_under_different_years(row):
    """For an unmatched row, return the sales year(s) available for
    this title, excluding the row's own year and any year already claimed
    by an exact match elsewhere for the same title.
    """
    if row["_merge"] == "both":
        return[]
    candidate_years = sales_years_by_title.get(row["title_norm"], [])
    claimed = matched_years_by_title.get(row["title_norm"], set())
    return [y for y in candidate_years if y != row["year"] and y not in claimed]


In [12]:
movies["_candidate_years"] = movies.apply(matching_years_under_different_years, axis=1)
year_mismatch = movies[movies["_candidate_years"].map(len) > 0].copy()
year_mismatch["sales_years_available"] = year_mismatch["_candidate_years"].apply(
    lambda ys: ", ".join(str(y) for y in ys)
)

1c. Corroborating two signals. Runtime and release-date-by-motnh/day are specific enough that two unrelated films sharing a title are unlikely to also share either by chance, and unlikely to share both.

In [13]:
RUNTIME_TOLERANCE_MIN = 2 #in minutes
_MONTHS = {m: i for i, m in enumerate(
    ["january", "february", "march", "april", "may", "june", "july", 
     "august", "september", "october", "november", "december"], start=1
)}


In [14]:
def parse_month_day(s):
    """Parse sales.xlsx's 'release_date' (e.g. 'May 6th') into a
    (month, day) tuple; returns None for the small percentage of values
    that don't follow that pattern (e.g. 'Summer')
    """
    if not isinstance(s, str):
        return None
    m = re.match(r'^([A-Za-z]+)\s+(\d{1,2})(st|nd|rd|th)?', s.strip())
    if not m:
        return None
    month = _MONTHS.get(m.group(1).lower())
    return (month, int(m.group(2))) if month else None

In [15]:
sales_dedup = sales_dedup.copy()
sales_dedup["_month_day"] = sales_dedup["release_date"].apply(parse_month_day)

In [16]:
sales_runtime_by_key = sales_dedup.set_index(["title_norm", "year"])["runtime"].to_dict()
sales_month_day_by_key = sales_dedup.set_index(["title_norm", "year"])["_month_day"].to_dict()
sales_genre_by_key = sales_dedup.set_index(["title_norm", "year"])["genre"].to_dict()

In [17]:
def is_documentary(genre_value):
    """True if the comma-separated genre string tags 'Documentary'.
    Case-insensitive; NaN/None safe.
    """
    if not isinstance(genre_value, str):
        return False
    return "documentary" in [g.strip().lower() for g in genre_value.split(",")]


In [18]:
MANUAL_MATCH_OVERRIDES = {
    ("17 blocks", 2021),
    ("wheel of time", 2003),
}

def match_evidence(row):
    """Compare MetaClean's runtime, release month/day, and (for the
    documentary case) genre against sales.xlxs's, for each remaining 
    candidate year. Returns a human-readable evidence string, a
    verdict, and the single matched year (NaN unless exactly one
    candidate year is accepted as a match).

    Each candidate year is classified on its own signals:
      - full match:         every available signal agrees            -> accept
      - date-corroborated:   release date matches, runtime does not   -> accept
      - documentary,
        one-year gap:        runtime matches, date does not, but BOTH
                              sides tag genre 'Documentary' and the
                              candidate year is exactly one year from
                              MetaClean's own year                    -> accept
      - manual override:     an explicitly named, human-verified pair -> accept
      - runtime-only:        runtime matches, date does not, none of
                              the above                                -> review
      - full mismatch:       every available signal disagrees          -> different
      - no data:             neither signal was available to compare

    None of this is a guess. A 12-row sample (6 date-corroborated, 6
    runtime-only) was checked against each film's own live Metacritic
    page (the real source MetaClean was built from). All 6
    date-corroborated cases were the same film, sales.xlsx recording an
    earlier festival/international release under its own year and a
    shorter cut's runtime. All 6 plain runtime-only cases did NOT hold
    up — sales.xlsx's date had no correspondence to the true release at
    all, so the runtime match alone was coincidental (round runtimes
    like 90 or 100 minutes recur across unrelated films far more than a
    specific date does), and every one of those 6 was a fiction genre
    (Drama, Action, Comedy), never Documentary. A further 5 documentary,
    one-year-gap rows, checked the same way (I Am Not Your Negro, After
    Parkland, Skid Row Marathon, Anthropocene: The Human Epoch, Speed
    Sisters), were ALL the same film, consistent with documentaries
    routinely having a festival release one year and a wide release the
    next, a gap fiction films don't structurally have. So the
    documentary + one-year-gap combination is treated as its own
    accepted category, distinct from — and better evidenced than —
    runtime-only agreement on its own; the exact genre match must be
    looked up by this row's own url/year, not by title alone, since two
    different films can share a title (e.g. MetaClean holds both a 2003
    comedy and a 2016 documentary both called "A Family Affair").
    """
    meta_rt, meta_md, meta_genre = row['runtime'], row['_month_day'], row['genre']
    meta_is_doc = is_documentary(meta_genre)
    parts = []
    accepted, runtime_only, full_mismatch, any_data = [], [], [], False
    for y in row['_candidate_years']:
        sales_rt = sales_runtime_by_key.get((row['title_norm'], y))
        sales_md = sales_month_day_by_key.get((row['title_norm'], y))
        sales_genre = sales_genre_by_key.get((row['title_norm'], y))
        rt_match, date_match = None, None
        bits = []
        if pd.notna(meta_rt) and sales_rt is not None and pd.notna(sales_rt):
            rt_match = abs(meta_rt - sales_rt) <= RUNTIME_TOLERANCE_MIN
            bits.append(f"runtime sales={sales_rt:.0f}min/meta={meta_rt:.0f}min ({'MATCH' if rt_match else 'differs'})")
        else:
            bits.append("runtime unavailable")
        if meta_md is not None and sales_md is not None:
            date_match = (meta_md == sales_md)
            bits.append(f"date sales={sales_md}/meta={meta_md} ({'MATCH' if date_match else 'differs'})")
        else:
            bits.append("date unavailable")

        is_manual_override = (row['title_norm'], y) in MANUAL_MATCH_OVERRIDES
        is_doc_one_year_gap = (
            meta_is_doc and is_documentary(sales_genre) and abs(row['year'] - y) == 1
        )
        if is_manual_override:
            bits.append("manually verified same film")
        elif is_doc_one_year_gap:
            bits.append("both sides tagged Documentary, 1-year gap")
        parts.append(f"{y}: " + ", ".join(bits))

        if is_manual_override:
            accepted.append(y)
            any_data = True
            continue

        signals = [s for s in (rt_match, date_match) if s is not None]
        if not signals:
            continue
        any_data = True
        if all(signals):
            accepted.append(y)
        elif date_match is True and rt_match is False:
            accepted.append(y)  # date-corroborated — verified pattern, see docstring
        elif rt_match is True and date_match is False:
            if is_doc_one_year_gap:
                accepted.append(y)  # documentary + 1-year gap — verified pattern, see docstring
            else:
                runtime_only.append(y)  # weaker — verified NOT reliable in general, see docstring
        else:
            full_mismatch.append(y)

    if len(accepted) == 1:
        verdict, matched_year = f"likely same film (year={accepted[0]})", accepted[0]
    elif len(accepted) > 1:
        verdict, matched_year = "ambiguous - multiple years match", float('nan')
    elif runtime_only:
        verdict, matched_year = "runtime matches but release date differs - needs review", float('nan')
    elif full_mismatch:
        verdict, matched_year = "likely different film", float('nan')
    else:
        verdict, matched_year = "no evidence available", float('nan')
    return pd.Series({
        'match_evidence': '; '.join(parts),
        'match_verdict': verdict,
        'match_year': matched_year,
    })

year_mismatch[['match_evidence', 'match_verdict', 'match_year']] = (
    year_mismatch.apply(match_evidence, axis=1)
)

In [19]:
auto_correct = year_mismatch[year_mismatch["match_year"].notna()].copy()
still_review = year_mismatch[year_mismatch["match_year"].isna()].copy()

dup = auto_correct.duplicated(subset=["title_norm", "match_year"], keep=False)
if dup.any():
    contested = auto_correct[dup].copy()
    contested["match_verdict"] = "contested - matched by multiple rows, needs manual review"
    still_review = pd.concat([still_review, contested], ignore_index=True)
    auto_correct = auto_correct[~dup]
    print(f"{dup.sum()} rows pulled back for review: same sales row matched by more than")

In [20]:
corrections = auto_correct[["movie_id", "title_norm", "match_year"]].merge(
    sales_dedup[["title_norm", "year", "production_budget", "worldwide_box_office"]],
    left_on=["title_norm", "match_year"], right_on=["title_norm", "year"],
    how="left"
)[["movie_id","production_budget","worldwide_box_office"]]

In [21]:
still_review_out = still_review[
    ['movie_id', 'title', 'url', 'year', 'sales_years_available', 'match_evidence', 'match_verdict']
].rename(columns={'year': 'meta_year', 'url': 'metacritic_url'})
still_review_out.to_csv(f"{OUT}/movies_year_mismatch_review.csv", index=False)
print(f"  {len(auto_correct)} rows auto-corrected into movies.csv on runtime + release-date evidence "
      f"(RUNTIME_TOLERANCE_MIN={RUNTIME_TOLERANCE_MIN} min)")
print(f"  movies_year_mismatch_review.csv: {len(still_review_out)} rows still needing manual verification")
verdict_counts = still_review_out['match_verdict'].str.replace(r' \(year=\d+\)', '', regex=True).value_counts()
for verdict, count in verdict_counts.items():
    print(f"    verdict '{verdict}': {count}")

  350 rows auto-corrected into movies.csv on runtime + release-date evidence (RUNTIME_TOLERANCE_MIN=2 min)
  movies_year_mismatch_review.csv: 225 rows still needing manual verification
    verdict 'likely different film': 143
    verdict 'runtime matches but release date differs - needs review': 82


In [22]:
def alnum_key(s):
    """Lowercase and strip every character that isn't a-z or 0-9. This
    removes ordinary punctuation and, as a side effect, the mojibake
    byte sequences documented for cast/director/title fields, since
    those are non-alphanumeric too."""
    if not isinstance(s, str):
        return ''
    return re.sub(r'[^a-z0-9]', '', s.lower())

In [23]:
meta['_title_alnum'] = meta['title'].apply(alnum_key)
movies['_title_alnum'] = meta['_title_alnum']  # meta and movies share row order/index
sales['_title_alnum'] = sales['title'].apply(alnum_key)

In [24]:
_alnum_year_titlenorms = sales.dropna(subset=['year']).groupby(['_title_alnum', 'year'])['title_norm'].nunique()
_ambiguous_alnum_keys = set(_alnum_year_titlenorms[_alnum_year_titlenorms > 1].index)

In [25]:
sales_alnum_dedup = (
    sales.dropna(subset=['year'])
    .loc[lambda d: ~d.set_index(['_title_alnum', 'year']).index.isin(_ambiguous_alnum_keys)]
    .drop_duplicates(subset=['_title_alnum', 'year'], keep='first')
)

In [26]:
unmatched_pool = movies[(movies['_merge'] == 'left_only') & (movies['_candidate_years'].map(len) == 0)]
# Select only movie_id/title/key columns from the left side before merging —
# unmatched_pool already carries production_budget/worldwide_box_office
# columns (all null, from the original join in section 1), and merging
# them alongside sales' same-named columns would collide and silently
# rename the sales-side values out from under this step instead of using
# them, which is exactly what happened on the first pass here.
alnum_matches = unmatched_pool[['movie_id', 'title', '_title_alnum', 'year']].merge(
    sales_alnum_dedup[['_title_alnum', 'year', 'title', 'production_budget', 'worldwide_box_office']],
    on=['_title_alnum', 'year'], how='inner', suffixes=('', '_sales')
)

In [27]:

dup_alnum = alnum_matches.duplicated(subset=['_title_alnum', 'year'], keep=False) | \
            alnum_matches.duplicated(subset=['movie_id'], keep=False)
if dup_alnum.any():
    print(f"  {dup_alnum.sum()} alnum matches skipped: same key claimed by more than one row")
    alnum_matches = alnum_matches[~dup_alnum]

In [28]:
alnum_audit = alnum_matches[['movie_id', 'title', 'title_sales', 'year', 'production_budget', 'worldwide_box_office']].rename(
    columns={'title': 'meta_title', 'title_sales': 'sales_title', 'year': 'matched_year'}
)
alnum_audit.to_csv(f"{OUT}/movies_alnum_matches.csv", index=False)


In [29]:
alnum_corrections = alnum_matches[['movie_id', 'production_budget', 'worldwide_box_office']]
movies = movies.merge(alnum_corrections, on='movie_id', how='left', suffixes=('', '_alnum'))
movies['production_budget'] = movies['production_budget_alnum'].combine_first(movies['production_budget'])
movies['worldwide_box_office'] = movies['worldwide_box_office_alnum'].combine_first(movies['worldwide_box_office'])
movies.drop(columns=['production_budget_alnum', 'worldwide_box_office_alnum'], inplace=True)

In [30]:
print(f"  movies_alnum_matches.csv: {len(alnum_audit)} rows auto-corrected on punctuation/encoding-tolerant "
      f"title + exact year match (spot-check this file — no independent corroborating signal like runtime/date)")


  movies_alnum_matches.csv: 344 rows auto-corrected on punctuation/encoding-tolerant title + exact year match (spot-check this file — no independent corroborating signal like runtime/date)


In [31]:
movies_out = movies[['movie_id', 'title', 'url', 'production_budget', 'worldwide_box_office', 'runtime']].rename(
    columns={'url': 'metacritic_url', 'worldwide_box_office': 'box_office', 'runtime': 'runtime_minutes'}
)

In [32]:
movies_out = movies_out.copy()
movies_out['runtime_minutes'] = to_int_str(movies_out['runtime_minutes'])
movies_out.to_csv(f"{OUT}/movies.csv", index=False)


In [33]:
n_matched = (movies['_merge'] == 'both').sum()
n_no_title_match_at_all = len(unmatched_pool)
n_still_no_match = n_no_title_match_at_all - len(alnum_matches)
print(f"  movies.csv: {len(movies_out)} rows")
print(f"    exact (title, year) match: {n_matched} ({n_matched/len(movies):.1%})")
print(f"    same title, different year, auto-corrected on runtime/date/genre evidence: {len(auto_correct)}")
print(f"    same title, different year, still needing review: {len(still_review)}")
print(f"    title not found under any year, auto-corrected on punctuation/encoding-tolerant match: {len(alnum_matches)}")
print(f"    title still not found under any year or spelling variant: {n_still_no_match}")
print(f"    both budget AND box office populated (H1 usable sample): "
      f"{(movies_out['production_budget'].notna() & movies_out['box_office'].notna()).sum()}")


  movies.csv: 11364 rows
    exact (title, year) match: 8009 (70.5%)
    same title, different year, auto-corrected on runtime/date/genre evidence: 350
    same title, different year, still needing review: 225
    title not found under any year, auto-corrected on punctuation/encoding-tolerant match: 344
    title still not found under any year or spelling variant: 2436
    both budget AND box office populated (H1 usable sample): 3322


In [34]:
print("\nDone. movies.csv is ready for pgAdmin's Import/Export tool.")
print("movies_year_mismatch_review.csv now holds only the rows that still need a team")
print("decision: runtime evidence pointed to a different film, was ambiguous between")
print("more than one candidate year, or was unavailable to decide with.")


Done. movies.csv is ready for pgAdmin's Import/Export tool.
movies_year_mismatch_review.csv now holds only the rows that still need a team
decision: runtime evidence pointed to a different film, was ambiguous between
more than one candidate year, or was unavailable to decide with.


In [35]:
import pandas as pd

movies_check = pd.read_csv("movies.csv")
print("Total rows:", len(movies_check))
print("Columns:", list(movies_check.columns))

both_populated = movies_check[
    movies_check["production_budget"].notna() & movies_check["box_office"].notna()
]
print("Rows with both budget and box office:", len(both_populated))
print(f"Share: {len(both_populated) / len(movies_check):.1%}")

print(movies_check["runtime_minutes"].dropna().astype(str).str.contains(r"\.0$").sum(), "rows still end in .0")

sample_titles = movies_check[movies_check["title"].str.contains("Am", na=False)]["title"]
print(sample_titles[sample_titles.str.contains("lie", na=False)])

Total rows: 11364
Columns: ['movie_id', 'title', 'metacritic_url', 'production_budget', 'box_office', 'runtime_minutes']
Rows with both budget and box office: 3322
Share: 29.2%
11109 rows still end in .0
618                 Amélie
679    Among the Believers
Name: title, dtype: str
